# 私榜因子人工复核

先运行配置 cell，再按需运行后面的检查。所有函数都会返回 DataFrame（或 DataFrame 字典），可继续在 notebook 中筛选分析。

In [ ]:
from pathlib import Path
import sys

# True: 本地；False: BigQuant 云端
USE_LOCAL = True

LOCAL_WORKSPACE_DIR = Path("/Users/xiehao/Desktop/workspace/BigAlpha")
CLOUD_WORKSPACE_DIR = Path("/home/aiuser/work/workspace/BigAlpha")
WORKSPACE_DIR = LOCAL_WORKSPACE_DIR if USE_LOCAL else CLOUD_WORKSPACE_DIR
COMPETITION_ID = "76ad3f56-ec2b-431a-890e-139a7f4bbcba"
FILES_DIR = (
    WORKSPACE_DIR / "system" / "files" / "private" / COMPETITION_ID
    if USE_LOCAL
    else WORKSPACE_DIR / "system" / "files" / COMPETITION_ID
)
PRIVATE_CODE_DIR = WORKSPACE_DIR / "system" / "competitions" / COMPETITION_ID / "private"
if str(PRIVATE_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(PRIVATE_CODE_DIR))

from manual_checks import *

paths = CheckPaths(
    run_dir=FILES_DIR / "private" / "runs" / "20260810_151358",
    prepared_dir=FILES_DIR / "private" / "prepared",
    private_code_dir=PRIVATE_CODE_DIR,
    bigalpha_eval_src=WORKSPACE_DIR / "eval" / "bigalpha_eval" / "src",
)

## 评分与排名

In [ ]:
score_problems = check_score_consistency(paths)
rank_conflicts = analyze_rank_conflicts(paths)
ab_sensitivity = analyze_ab_weight_sensitivity(paths)
a_metric_sensitivity = analyze_a_metric_sensitivity(paths)

## 回归产物与 B 分稳健性

In [ ]:
regression_integrity = check_regression_integrity(paths)
regression_stability = analyze_regression_stability(paths)
b_score_robustness = analyze_b_score_robustness(paths)

## 因子相似度

## 可视化

第一项只读取现有 CSV；第二项会完整复跑滚动回归，仅在 BigQuant 评测环境中按需运行。

In [ ]:
regression_overview = plot_regression_overview(paths)

In [ ]:
# 耗时操作：需要 dai / bigmodule 和正式行情数据
# 在云端运行，并将整个 regression_rerun 目录下载到本地 run/artifacts 下
# regression_explanation = rerun_regression_explanation(
#     paths,
#     sd="2025-01-01",
#     ed="2026-08-10",
#     output_dir=paths.artifacts_dir / "regression_rerun",
#     tolerance=1e-8,
#     plot=True,
# )

## 一键生成 Markdown 报告

修改配置 cell 中的 `run_dir` 后运行本 cell。报告默认生成到当前 run 的 `artifacts/manual_check_report.md`。

In [ ]:
report_path = generate_markdown_report(paths)
print(report_path)